# Week 5 Senpai RAG Optimisation - Atomic-Section Historical Run

> **Status: SUPERSEDED / HISTORICAL v1.0.0.** This notebook is retained only for provenance. It used pre-segmented atomic-section parents, so the registered chunk sizes did not change the actual candidate inputs. Use `W05_RAG_Optimisation.ipynb` at the standard deliverable path for the latest long-source corrective result.

This executed notebook presents the registered Week 5 full-factorial result:
3 chunk sizes × 3 top-k values × reranking off/cross-encoder, with 20 frozen
Senpai questions per configuration (360 rows). All quality values are local,
uncalibrated diagnostics; latency is a one-A40 warm-path measurement and is not
a deployed-product benchmark.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

cwd = Path.cwd()
HERE = cwd / "phase_c_synthesis" if (cwd / "phase_c_synthesis").exists() else cwd
RESULT_PATH = HERE / "W05_RAG_Optimisation_Summary_v1.0.0.json"
SUMMARY_PATH = HERE / "W05_RAG_Optimisation_Summary_v1.0.0.csv"
ITEM_PATH = HERE / "W05_RAG_Optimisation_Item_Results_v1.0.0.csv"

result = json.loads(RESULT_PATH.read_text(encoding="utf-8"))
summary = pd.read_csv(SUMMARY_PATH)
items = pd.read_csv(ITEM_PATH)
contrasts = pd.DataFrame(result["matched_contrasts"])

assert len(summary) == 18
assert len(items) == 360
assert items["variant_id"].nunique() == 18
assert (items.groupby("variant_id").size() == 20).all()
assert len(contrasts) == 45

pd.DataFrame({
    "check": ["factorial variants", "rows", "items per variant", "matched contrasts"],
    "observed": [len(summary), len(items), "20 for all 18", len(contrasts)],
})

,check,observed
0,factorial variants,18
1,rows,360
2,items per variant,20 for all 18
3,matched contrasts,45


## Registered comparison logic

The full factorial holds the candidate model, prompts, 20 questions, seed and
scorers fixed while varying three retrieval factors. Each matched contrast
changes exactly one factor. Pareto dominance maximizes mean Faithfulness and
weighted required-point Coverage while minimizing median warm-path
question-to-response latency. Relevance is reported but is not a Pareto axis.

In [2]:
pareto_ids = set(result["pareto_variant_ids"])
view = summary.copy()
view["pareto_optimal"] = view["variant_id"].isin(pareto_ids)
view[[
    "variant_id", "mean_faithfulness", "faithfulness_coverage",
    "mean_answer_relevance", "mean_required_point_coverage",
    "p50_question_to_response_ms", "p95_question_to_response_ms",
    "pareto_optimal",
]].sort_values(
    ["pareto_optimal", "mean_faithfulness", "mean_required_point_coverage"],
    ascending=[False, False, False],
).reset_index(drop=True)

,variant_id,mean_faithfulness,faithfulness_coverage,mean_answer_relevance,mean_required_point_coverage,p50_question_to_response_ms,p95_question_to_response_ms,pareto_optimal
0,chunk-512_topk-3_rerank-ce,0.933709,0.95,0.639153,0.886548,5579.3110,9180.04135,True
1,chunk-256_topk-5_rerank-none,0.907500,1.00,0.631559,0.932500,7145.6145,11518.55610,True
2,chunk-1024_topk-1_rerank-ce,0.869048,0.90,0.437968,0.766786,3050.2715,7122.01220,True
3,chunk-512_topk-5_rerank-ce,0.849405,1.00,0.605984,0.969167,7794.8615,11921.63070,True
4,chunk-512_topk-1_rerank-none,0.849266,0.90,0.370510,0.766786,2781.2315,6716.76880,True
5,chunk-1024_topk-3_rerank-ce,0.933709,0.95,0.639153,0.886548,5597.7410,9206.78515,False
6,chunk-256_topk-3_rerank-ce,0.933709,0.95,0.639153,0.886548,5650.9385,9161.12160,False
7,chunk-1024_topk-5_rerank-none,0.907500,1.00,0.631559,0.932500,7158.4170,11521.77525,False
8,chunk-512_topk-5_rerank-none,0.907500,1.00,0.631559,0.932500,7158.7630,11613.58695,False
9,chunk-1024_topk-3_rerank-none,0.885088,0.95,0.570124,0.849167,5657.4380,8846.67010,False


## Pareto set and transparent balanced choice

In [3]:
frontier = view[view["pareto_optimal"]][[
    "variant_id", "mean_faithfulness", "mean_required_point_coverage",
    "mean_answer_relevance", "p50_question_to_response_ms",
]].sort_values("p50_question_to_response_ms")
display(frontier.reset_index(drop=True))
display(pd.DataFrame([result["balanced_choice"]]))

,variant_id,mean_faithfulness,mean_required_point_coverage,mean_answer_relevance,p50_question_to_response_ms
0,chunk-512_topk-1_rerank-none,0.849266,0.766786,0.370510,2781.2315
1,chunk-1024_topk-1_rerank-ce,0.869048,0.766786,0.437968,3050.2715
2,chunk-512_topk-3_rerank-ce,0.933709,0.886548,0.639153,5579.3110
3,chunk-256_topk-5_rerank-none,0.907500,0.932500,0.631559,7145.6145
4,chunk-512_topk-5_rerank-ce,0.849405,0.969167,0.605984,7794.8615


,variant_id,quality_harmonic_mean,selection_rule
0,chunk-256_topk-5_rerank-none,0.91983,highest harmonic mean of diagnostic Faithfulne...


## Matched contrasts

Positive quality deltas favor the right-hand factor level; positive latency
deltas mean the right-hand level is slower. These are within-design descriptive
contrasts, not universal causal effects outside the frozen corpus and stack.

In [4]:
contrast_metrics = [
    "mean_delta_faithfulness", "mean_delta_answer_relevance",
    "mean_delta_required_point_coverage",
    "mean_delta_question_to_response_ms",
]
display(
    contrasts.groupby("factor")[contrast_metrics]
    .agg(["count", "mean", "min", "max"])
    .round(6)
)
display(pd.DataFrame(result["reranking_interaction_by_top_k"]))
display(contrasts.sort_values(["factor", "left_variant_id"]).reset_index(drop=True))

mean_delta_faithfulness                                \
                                    count      mean       min       max   
factor                                                                    
chunk_size_tokens                      18  0.000000  0.000000  0.000000   
reranking                               9  0.003436 -0.058095  0.048622   
top_k                                  18  0.006403 -0.092231  0.060979   

                  mean_delta_answer_relevance                                \
                                        count      mean       min       max   
factor                                                                        
chunk_size_tokens                          18  0.000000  0.000000  0.000000   
reranking                                   9  0.036971 -0.025574  0.069029   
top_k                                      18  0.143021 -0.033168  0.261048   

                  mean_delta_required_point_coverage                      \
                                               count      mean       min   
factor                                                                     
chunk_size_tokens                                 18  0.000000  0.000000   
reranking                                          9  0.024683  0.000000   
top_k                                             18  0.122698  0.082381   

                            mean_delta_question_to_response_ms               \
                        max                              count         mean   
factor                                                                        
chunk_size_tokens  0.000000                                 18   -17.015572   
reranking          0.037381                                  9   482.442567   
top_k              0.202381                                 18  2298.323739   

                                           
                          min         max  
factor                                     
chunk_size_tokens   -62.87800    36.72865  
reranking           -32.00315  1054.60705  
top_k              1217.53610  3754.75940

,fixed_top_k,contrasts,mean_delta_faithfulness,mean_delta_answer_relevance,mean_delta_required_point_coverage,mean_delta_question_to_response_ms
0,1,3,0.019781,0.067458,0.000000,439.450667
1,3,3,0.048622,0.069029,0.037381,-21.585717
2,5,3,-0.058095,-0.025574,0.036667,1029.462750


,factor,left_variant_id,right_variant_id,left_value,right_value,fixed_chunk_size_tokens,fixed_top_k,fixed_reranking,mean_delta_faithfulness,paired_rows_faithfulness,mean_delta_answer_relevance,paired_rows_answer_relevance,mean_delta_required_point_coverage,paired_rows_required_point_coverage,mean_delta_question_to_response_ms,paired_rows_question_to_response_ms
0,chunk_size_tokens,chunk-256_topk-1_rerank-ce,chunk-1024_topk-1_rerank-ce,256,1024,NaN,1.0,cross_encoder,0.000000,18,0.000000,20,0.000000,20,-26.14935,20
1,chunk_size_tokens,chunk-256_topk-1_rerank-ce,chunk-512_topk-1_rerank-ce,256,512,NaN,1.0,cross_encoder,0.000000,18,0.000000,20,0.000000,20,36.72865,20
2,chunk_size_tokens,chunk-256_topk-1_rerank-none,chunk-1024_topk-1_rerank-none,256,1024,NaN,1.0,none,0.000000,18,0.000000,20,0.000000,20,-48.98560,20
3,chunk_size_tokens,chunk-256_topk-1_rerank-none,chunk-512_topk-1_rerank-none,256,512,NaN,1.0,none,0.000000,18,0.000000,20,0.000000,20,-51.55515,20
4,chunk_size_tokens,chunk-256_topk-3_rerank-ce,chunk-1024_topk-3_rerank-ce,256,1024,NaN,3.0,cross_encoder,0.000000,19,0.000000,20,0.000000,20,0.03465,20
5,chunk_size_tokens,chunk-256_topk-3_rerank-ce,chunk-512_topk-3_rerank-ce,256,512,NaN,3.0,cross_encoder,0.000000,19,0.000000,20,0.000000,20,11.62850,20
6,chunk_size_tokens,chunk-256_topk-3_rerank-none,chunk-1024_topk-3_rerank-none,256,1024,NaN,3.0,none,0.000000,19,0.000000,20,0.000000,20,-13.13415,20
7,chunk_size_tokens,chunk-256_topk-3_rerank-none,chunk-512_topk-3_rerank-none,256,512,NaN,3.0,none,0.000000,19,0.000000,20,0.000000,20,-6.45500,20
8,chunk_size_tokens,chunk-256_topk-5_rerank-ce,chunk-1024_topk-5_rerank-ce,256,1024,NaN,5.0,cross_encoder,0.000000,20,0.000000,20,0.000000,20,-36.66015,20
9,chunk_size_tokens,chunk-256_topk-5_rerank-ce,chunk-512_topk-5_rerank-ce,256,512,NaN,5.0,cross_encoder,0.000000,20,0.000000,20,0.000000,20,10.43620,20


## Coverage, structural identity and provenance

In [5]:
display(pd.DataFrame([
    {"metric": key, "finite_fraction": value}
    for key, value in result["metric_coverage"].items()
]))
display(pd.DataFrame([result["scoring_audit"]]))
display(pd.DataFrame([result["chunk_level_identity"]]))
display(pd.DataFrame([
    {"source": name, **metadata}
    for name, metadata in result["source_files"].items()
]))

,metric,finite_fraction
0,faithfulness,0.95
1,answer_relevance,1.00
2,required_point_coverage,1.00
3,latency,1.00


,ragas_status_counts,coverage_status_counts,ragas_scorer_versions,coverage_scorer_versions,ragas_reused_rows,coverage_reused_rows
0,"{'complete': 342, 'metric_failure_retained': 18}",{'parsed': 360},"[1.0.0, 1.1.0]",[1.3.0],260,269


,three_chunk_level_matched_groups,identical_candidate_message_groups,identical_candidate_output_groups,message_identity_rate,output_identity_rate,interpretation
0,120,120,120,1.0,1.0,High identity means the atomic source sections...


,source,sha256,rows
0,generations,4955084460ed1572511835457638e46111855ea5397d72...,360
1,ragas,4ed8164a9abb58e8d43f82eccd046b1ca480bff6c1a26d...,360
2,coverage,0de38ab1cdd54c4fd57c0910902f7e2725a1d26f4dd473...,360


## Interpretation boundary

The balanced choice is a transparent diagnostic recommendation within the
observed Pareto set, not a production-readiness claim. RAGAS scores were not
human calibrated. Faithfulness is retained as missing where no claim could be
extracted rather than imputed. High cross-chunk output identity indicates a
small, atomically sectioned corpus and limits conclusions about chunk size.
Randomized block order and warm/cold separation reduce identifiable run-order
and initialization confounding, but one run cannot establish external
causality or a universal mechanism.